# 01 — EDA MIMIC-III

**Bloqueado**: requer uma conta credenciada no PhysioNet (ver `data/download_mimic.py`) e rodar `python data/merge_mimic_tables.py` depois do download. Sem isso, as células abaixo falham com `FileNotFoundError` — o código está pronto para quando os dados existirem, mas esta notebook não foi executada (nenhum output foi gerado).

Enquanto isso, os experimentos e notebooks 02-08 rodam sobre MNIST/CIFAR-10 non-IID (`data/download_datasets.py`), já disponíveis.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

MIMIC_DIR = '../data/raw/mimic'
os.makedirs('../results/figures', exist_ok=True)

admissions = pd.read_csv(f'{MIMIC_DIR}/ADMISSIONS.csv')
patients = pd.read_csv(f'{MIMIC_DIR}/PATIENTS.csv')
diagnoses = pd.read_csv(f'{MIMIC_DIR}/DIAGNOSES_ICD.csv')

admissions['ADMITTIME'] = pd.to_datetime(admissions['ADMITTIME'])
admissions['DISCHTIME'] = pd.to_datetime(admissions['DISCHTIME'])
patients['DOB'] = pd.to_datetime(patients['DOB'])

print(f'Total pacientes: {len(patients)}')
print(f'Total admissões: {len(admissions)}')
print(f'Admissões por paciente: {len(admissions) / len(patients):.2f}')

In [ ]:
# Idade na admissão (ver nota sobre deslocamento de datas >89 anos em
# data/merge_mimic_tables.py)
df = admissions.merge(patients[['SUBJECT_ID', 'DOB']], on='SUBJECT_ID', how='left')
df['age'] = (df['ADMITTIME'] - df['DOB']).dt.days / 365.25

print(f"Idade média: {df['age'].mean():.1f} anos")
print(f"Idade range: {df['age'].min():.1f} - {df['age'].max():.1f} anos")

In [ ]:
def calc_readmission_30d(patient_admissions):
    patient_admissions = patient_admissions.sort_values('ADMITTIME')
    flags = []
    for i in range(len(patient_admissions) - 1):
        gap_days = (patient_admissions.iloc[i + 1]['ADMITTIME'] -
                    patient_admissions.iloc[i]['DISCHTIME']).days
        flags.append(1 if 0 < gap_days <= 30 else 0)
    return np.mean(flags) if flags else 0.0

sample_ids = admissions['SUBJECT_ID'].unique()[:1000]
rates = [calc_readmission_30d(admissions[admissions['SUBJECT_ID'] == pid]) for pid in sample_ids]
print(f'Taxa média de readmissão em 30 dias (amostra de {len(sample_ids)} pacientes): '
      f'{np.mean(rates):.1%}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].hist(df['age'].dropna(), bins=50, edgecolor='black')
axes[0, 0].set_title('Distribuição de idades na admissão')
axes[0, 0].set_xlabel('Idade (anos)')

comorbidities = diagnoses.groupby('HADM_ID').size()
axes[0, 1].hist(comorbidities, bins=50, edgecolor='black')
axes[0, 1].set_title('Distribuição de comorbidades por admissão')
axes[0, 1].set_xlabel('Número de diagnósticos')

axes[1, 0].hist(rates, bins=20, edgecolor='black')
axes[1, 0].set_title('Distribuição de readmissão em 30 dias (por paciente)')
axes[1, 0].set_xlabel('Taxa de readmissão')

admission_types = admissions['ADMISSION_TYPE'].value_counts()
axes[1, 1].barh(admission_types.index, admission_types.values)
axes[1, 1].set_title('Tipo de admissão')

plt.tight_layout()
plt.savefig('../results/figures/01_mimic_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## Próximo passo

Rode `python data/merge_mimic_tables.py` para gerar `merged_data.csv`, e então `python src/utils/hospital_splitter.py` para produzir os 5 splits de hospitais non-IID em `data/processed/hospital_{a-e}/`.